# 09a THINGS-EEG2 test P-4: the personal map on a second dataset (prespecified, scored once)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset
`cross-subject-brain-encoding-eeg` (09g) if attached; otherwise the release is downloaded from OSF.

Protocol P-4 (`docs/proposals/eeg2_h4b_replication_protocol.md`). Each of the 10 people in turn is the new person and
borrows from the other 9. The personal map (borrowing plus a 17 x 17 channel map per 100 ms window, fitted on k
labelled images) against six comparators: borrowing, Gu's ensemble, the converter, own ridge, own ridge with an
output map, and the generic map. k = 250 and 500, draws 13 to 15, 100 to 200 ms, margin 0.011, Bonferroni over six
at alpha 0.05, all 10 people required.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09a-h4beeg2', 'cpu-confirm'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"people": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09", "sub-10"], "budgets": [250, 500], "draws": [13, 14, 15], "window_s": [0.1, 0.2], "map_window_ms": 100, "delta": 0.011, "alpha": 0.05, "min_targets": 10, "candidate": "h4b", "comparators": ["avg_sources", "gu_window", "linear_converter", "own_ridge", "own_ridge_oc", "generic"], "protocol": "P-4 frozen: H4b replication on THINGS-EEG2, new family"}')
import shutil
import numpy as np
import pandas as pd

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.data.splits import subset
from cross_subject_encoding.eval import eeg1_decision as dec
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.baselines import best_source, gu_by_group, window_groups
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset
from cross_subject_encoding.utils.preflight import preflight

preflight("cross-subject-brain-encoding-09a-outputs", inputs=["paper_alexnet_kpca1000.npz", C.EEG_K_SUBSETS],
          need_gpu=False, expect_output_gb=0.1)
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
people = FIXED["people"]
ATTACHED = C.EEG_RAW.is_dir() and any(C.EEG_RAW.rglob("preprocessed_eeg_test.npy"))
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test = {}, {}
for s in people:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s] = tr["data"].mean(1).astype(np.float32)  # (images, channels, times), 4 repetitions averaged
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"].astype(np.float32)
    times = tr["times"]
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
    print(s, "loaded |", f"{(time.monotonic() - T0) / 60:.1f} min")
N_TRAIN, N_TEST = len(z_train), len(z_test)
n_ch, n_t = bio_test[people[0]].shape[2:]
assert (n_ch, n_t) == (17, 100) and np.isclose(times[0], -0.2)
query = np.concatenate([z_train, z_test])
src_train = np.empty((len(people), N_TRAIN, n_ch * n_t), np.float32)
src_test = np.empty((len(people), N_TEST, n_ch * n_t), np.float32)
for i, s in enumerate(people):  # each person's own full-data encoder (the paper's OLS)
    pred = pl.ols_paper(z_train, y_train[s].reshape(N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:N_TRAIN], pred[N_TRAIN:]
groups = window_groups(times, n_ch, C.EEG_WINDOW_MS)
map_windows = np.round((times - times[0]) * 1000).astype(int) // FIXED["map_window_ms"]
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
orders = {(s, d, k): np.asarray(subset(subsets, s, d, k)) for s in people for d in FIXED["draws"] for k in FIXED["budgets"]}
print("sources fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
ALPHAS = [10.0 ** e for e in range(-2, 9)]
rows, failed, done = [], {}, []
for i, person in enumerate(people):
    t = time.monotonic()
    try:
        others = [j for j in range(len(people)) if j != i]
        avg_train, avg_test = src_train[others].mean(0), src_test[others].mean(0)
        avg_t3 = avg_test.reshape(N_TEST, n_ch, n_t)
        preds = {"avg_sources|all|all": avg_t3}
        for d in FIXED["draws"]:
            for k in FIXED["budgets"]:
                idx = orders[(person, d, k)]
                target = y_train[person][idx]
                flat = target.reshape(k, -1)
                own_map, _ = h4b.fit_map(avg_train[idx].reshape(k, n_ch, n_t), target, map_windows, seed=d)
                preds[f"h4b|{k}|{d}"] = h4b.apply_map(own_map, avg_t3, map_windows)
                maps = []
                for q in others:  # generic: each other person's map, never using this person's data
                    rest = [j for j in others if j != q]
                    idx_q = orders[(people[q], d, k)]
                    avg_q = src_train[np.ix_(rest, idx_q)].mean(0).reshape(k, n_ch, n_t)
                    maps.append(h4b.fit_map(avg_q, y_train[people[q]][idx_q], map_windows, seed=d)[0])
                preds[f"generic|{k}|{d}"] = h4b.apply_map(h4b.mean_map(maps), avg_t3, map_windows)
                own = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(z_train[idx], flat)
                own_test = own.predict(z_test)
                oof = np.empty_like(flat)
                for fold in np.array_split(np.random.default_rng(d).permutation(k), C.RIDGE_FOLDS):
                    keep = np.setdiff1d(np.arange(k), fold)
                    oof[fold] = own.refit(z_train[idx[keep]], flat[keep]).predict(z_train[idx[fold]])
                preds[f"own_ridge|{k}|{d}"] = own_test
                preds[f"own_ridge_oc|{k}|{d}"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(oof, flat).predict(own_test)
                borrowed = src_train[np.ix_(others, idx)]
                best = best_source(list(borrowed), flat)
                preds[f"linear_converter|{k}|{d}"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(borrowed[best], flat).predict(src_test[others][best])
                preds[f"gu_window|{k}|{d}"] = gu_by_group(borrowed, flat, src_test[others], groups, "pooled")
        preds = {n: np.asarray(p).reshape(N_TEST, n_ch, n_t) for n, p in preds.items()}
        if not all(np.isfinite(p).all() for p in preds.values()):
            raise ValueError("nonfinite prediction")
        scored = paper_correlation(preds, bio_test[person], n_iter=100)
        for name, curve in scored["correlation"].items():
            method, k, d = name.split("|")
            rows.append({"person": person, "method": method, "k": k, "draw": d, "r_100_200ms": float(curve[:, window].mean())})
        done.append(person)
    except Exception as error:  # P-4: a failed person is excluded and reported, never imputed
        failed[person] = f"{type(error).__name__}: {error}"
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "09a_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s", failed.get(person, ""))
frame = pd.DataFrame(rows)
mean = frame.groupby(["person", "method", "k"])["r_100_200ms"].mean()
scores = {}
for (person, method, k), value in mean.items():
    for b in FIXED["budgets"]:
        if k == "all" or k == str(b):
            scores[(person, method, b)] = value
comparators = tuple(FIXED["comparators"])
d = dec.endpoint(scores, people, FIXED["budgets"], comparators=comparators, candidate=FIXED["candidate"])
decision = dec.decide(d, FIXED["delta"], FIXED["alpha"], FIXED["min_targets"], comparators=comparators)
decision["sign_flip_p"] = {c: dec.sign_flip_p(d[:, j]) for j, c in enumerate(comparators)}
decision["d"] = {p: row.tolist() for p, row in zip(people, d)}
decision["failed"] = failed
decision["method_means"] = {f"{m}|{b}": float(np.nanmean([scores.get((p, m, b), np.nan) for p in done]))
                            for m in (FIXED["candidate"],) + comparators for b in FIXED["budgets"]}
(C.METRICS_DIR / "09a_decision.json").write_text(json.dumps(decision, indent=2))
print("VERDICT:", decision["verdict"], "| people", decision["n_targets"], "| failed", len(failed))
print(json.dumps(decision["per_comparator"], indent=1))

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-09a-outputs", title="XSUB 09a H4b EEG2 replication outputs",
                  files=[C.METRICS_DIR / "09a_decision.json", C.METRICS_DIR / "09a_scores.csv"], include_dirs=[],
                  message="h4b eeg2 replication")